# 0: Read all TXT data from ODS tables

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("CombineTXTData").getOrCreate()

# Read good TXT data
df_txt_good = spark.table("ODS.Employee_Text_Data")

# Read TXT error data (if you want to include corrected data)
df_txt_errors = spark.table("ODS.Employee_Text_Data_Errors")

StatementMeta(, 51b91c34-088e-474b-8c39-afc2d171217f, 5, Finished, Available, Finished, False)

# 1: Analyze Error Types

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, lit, split, size, expr, desc

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("AnalyzeTXTERrors").getOrCreate()

# Read error data
df_txt_errors = spark.table("ODS.Employee_Text_Data_Errors")

# Analyze error distribution
print("="*50)
print("TXT ERROR ANALYSIS")
print("="*50)

print("\n📊 Error Type Distribution:")
df_txt_errors.groupBy("error_type").count().orderBy(desc("count")).show()

print("\n📊 Error Details:")
# Separate errors by type
df_missing = df_txt_errors.filter(col("error_type") == "MISSING_COLUMNS")
df_extra = df_txt_errors.filter(col("error_type") == "EXTRA_COLUMNS")

print(f"Missing columns errors: {df_missing.count()}")
print(f"Extra columns errors: {df_extra.count()}")

StatementMeta(, 51b91c34-088e-474b-8c39-afc2d171217f, 6, Finished, Available, Finished, False)

TXT ERROR ANALYSIS

📊 Error Type Distribution:


+---------------+-----+
|     error_type|count|
+---------------+-----+
|  EXTRA_COLUMNS|    5|
|MISSING_COLUMNS|    2|
+---------------+-----+


📊 Error Details:
Missing columns errors: 2
Extra columns errors: 5


# 2: Investigate Missing Columns Errors

In [9]:
print("="*50)
print("MISSING COLUMNS ERRORS")
print("="*50)

# Show sample of missing column errors
print("\n🔍 Sample Missing Column Errors:")
display(df_missing.limit(10).toPandas())

# Analyze missing column patterns
print("\n📊 Missing Column Patterns:")
df_missing.select(
    col("expected_fields"),
    col("actual_fields"),
    col("original_line")
).limit(20).show(truncate=False)

# Calculate difference between expected and actual
df_missing = df_missing.withColumn(
    "missing_count",
    col("expected_fields").cast("int") - col("actual_fields").cast("int")
)

print("\n📊 Number of missing fields per row:")
df_missing.groupBy("missing_count").count().orderBy("missing_count").show()

StatementMeta(, 51b91c34-088e-474b-8c39-afc2d171217f, 7, Finished, Available, Finished, False)

MISSING COLUMNS ERRORS

🔍 Sample Missing Column Errors:


SynapseWidget(Synapse.DataFrame, 96ed292c-e3d1-45bd-a25c-459abd9b939b)


📊 Missing Column Patterns:
+---------------+-------------+-------------------------------------------------------------------------------------------+
|expected_fields|actual_fields|original_line                                                                              |
+---------------+-------------+-------------------------------------------------------------------------------------------+
|12             |11           |100250|Heba Mostafa|Heba|Mostafa||SALES|Jordan|44676|Male|Supply Chain Coordinator|on leave|
|12             |11           |100501|Layla Miller|Layla|Miller|14/08/1995|PROD| uk |12250.75|M|Accountant|A              |
+---------------+-------------+-------------------------------------------------------------------------------------------+


📊 Number of missing fields per row:


+-------------+-----+
|missing_count|count|
+-------------+-----+
|            1|    2|
+-------------+-----+



### Issue 1: In line 252 or id of 100250 the date of birth is empty (not the main issue) and education column is missing (main issue) 
### Issue 2: In line 502 or id of 100501 the employee_status is not logical (Does not have a logical or useful value) and education is missing

# 3: Solve Missing Columns Errors

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("FixMissingColumns").getOrCreate()

print("="*50)
print("DIRECT FIX FOR SPECIFIC MISSING COLUMNS")
print("="*50)

# Current timestamp
current_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Define schema explicitly to handle None values
schema = StructType([
    StructField("emp_id", StringType(), True),
    StructField("full_name", StringType(), True),
    StructField("fname", StringType(), True),
    StructField("lname", StringType(), True),
    StructField("birth_date", StringType(), True),
    StructField("dept_name", StringType(), True),
    StructField("country", StringType(), True),
    StructField("monthly_salary", StringType(), True),
    StructField("sex", StringType(), True),
    StructField("job_title", StringType(), True),
    StructField("employee_status", StringType(), True),
    StructField("education", StringType(), True),
    StructField("load_timestamp", StringType(), True),
    StructField("source_location", StringType(), True),
    StructField("file_type", StringType(), True),
    StructField("was_error", StringType(), True)
])

# Create the fixed rows manually based on the error records
fixed_data = []

# Row 1: 100250 - Missing birth_date
fixed_data.append({
    'emp_id': '100250',
    'full_name': 'Heba Mostafa',
    'fname': 'Heba',
    'lname': 'Mostafa',
    'birth_date': None,
    'dept_name': 'SALES',
    'country': 'Jordan',
    'monthly_salary': '44676',
    'sex': 'Male',
    'job_title': 'Supply Chain Coordinator',
    'employee_status': 'on leave',
    'education': None,
    'load_timestamp': current_timestamp,
    'source_location': 'Tables/ODS/employee_text_data_errors',
    'file_type': 'txt',
    'was_error': 'MISSING_FIXED'
})

# Row 2: 100501 - employee_status is 'A' (invalid), education missing
fixed_data.append({
    'emp_id': '100501',
    'full_name': 'Layla Miller',
    'fname': 'Layla',
    'lname': 'Miller',
    'birth_date': '14/08/1995',
    'dept_name': 'PROD',
    'country': 'uk',
    'monthly_salary': '12250.75',
    'sex': 'M',
    'job_title': 'Accountant',
    'employee_status': None,
    'education': None,
    'load_timestamp': current_timestamp,
    'source_location': 'Tables/ODS/employee_text_data_errors',
    'file_type': 'txt',
    'was_error': 'MISSING_FIXED'
})

# Convert to DataFrame with explicit schema
df_fixed_manual = spark.createDataFrame(fixed_data, schema)

print("\n✅ Manually fixed rows:")
display(df_fixed_manual)


# Show the fixed rows
print("\n📊 Fixed rows details:")
display(df_fixed_manual.select(
    "emp_id",
    "full_name",
    "birth_date",
    "employee_status",
    "education",
    "was_error"
).toPandas())

StatementMeta(, 51b91c34-088e-474b-8c39-afc2d171217f, 8, Finished, Available, Finished, False)

DIRECT FIX FOR SPECIFIC MISSING COLUMNS

✅ Manually fixed rows:


SynapseWidget(Synapse.DataFrame, 893cae6e-b208-4211-9885-ebaf978d35eb)


📊 Fixed rows details:


SynapseWidget(Synapse.DataFrame, 0a2f2a13-0478-4cac-ab2d-797a73092bd2)

# 4: Investigate Extra Columns Errors

In [11]:
print("="*50)
print("EXTRA COLUMNS ERRORS")
print("="*50)

# Show sample of extra column errors
print("\n🔍 Sample Extra Column Errors:")
display(df_extra.limit(10).toPandas())

# Analyze extra column patterns
print("\n📊 Extra Column Patterns:")
df_extra.select(
    col("expected_fields"),
    col("actual_fields"),
    col("original_line")
).limit(20).show(truncate=False)

df_extra = df_extra.withColumn(
    "extra_count",
    col("actual_fields").cast("int") - col("expected_fields").cast("int")
)

print("\n📊 Number of extra fields per row:")
df_extra.groupBy("extra_count").count().orderBy("extra_count").show()

StatementMeta(, 51b91c34-088e-474b-8c39-afc2d171217f, 9, Finished, Available, Finished, False)

EXTRA COLUMNS ERRORS

🔍 Sample Extra Column Errors:


SynapseWidget(Synapse.DataFrame, ae3fef8f-1b3c-4240-9495-2b0e4334320b)


📊 Extra Column Patterns:


+---------------+-------------+-----------------------------------------------------------------------------------------------------------------------+
|expected_fields|actual_fields|original_line                                                                                                          |
+---------------+-------------+-----------------------------------------------------------------------------------------------------------------------+
|12             |13           |100777|Abdullah Davis|Abdullah|Davis|1996-12-05|human resources|QAT|14131|Male|Project Manager|External| PhD |EXTRA_COL|
|12             |13           |100611|Smith, Emily|Emily|Smith|12-26-2000|Operations|EG|34,909|female|Production Operator|Level 2|Terminated|PhD      |
|12             |13           |100889|Johnson, Noora|Noora|Johnson|1974-02-05| HR |SA|N/A|female|Business Analyst|Level 2|Probation|bachelor          |
|12             |13           |100125|Al-Kuwari, Aya|Aya|Al-Kuwari|09-28-1967|Production

### Issue 3: row with id 100777 has an extra column at the end
### Issue 4: row with id 100611 has an extra column between job title and employee status and it contains Level 2 value
### Issue 5: row with id 100889 has an extra column between job title and employee status and it contains Level 2 value
### Issue 6: row with id 100125 has an extra column between job title and employee status and it contains Level 2 value
### Issue 7: row with id 100333 has an extra column at the end

# 5: Solving Extra Columns Errors

In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, StringType

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("FixTXTErrors").getOrCreate()

df_extra = spark.table("ODS.Employee_Text_Data_Errors").filter(col("error_type") == "EXTRA_COLUMNS")

expected_cols = ['emp_id', 'full_name', 'fname', 'lname', 'birth_date', 'dept_name', 'country', 'monthly_salary', 'sex', 'job_title', 'employee_status', 'education']

current_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def fix_extra_columns(df_errors):
    fixed_rows = []
    for row in df_errors.collect():
        original_line = row['original_line']
        fields = original_line.split('|')
        
        # Check for extra column at the end
        if fields[-1].strip() in ['EXTRA_COL']:
            fields = fields[:-1]
        
        # Check for 'Level 2' between job_title and employee_status
        if len(fields) > 12:
            for i in range(len(fields) - 1):
                if fields[i] and fields[i].strip() == 'Level 2':
                    fields.pop(i)
                    break
        
        # Ensure exactly 12 columns
        if len(fields) > 12:
            fields = fields[:12]
        elif len(fields) < 12:
            while len(fields) < 12:
                fields.append(None)
        
        fixed_row = dict(zip(expected_cols, fields))
        fixed_row['load_timestamp'] = current_timestamp
        fixed_row['source_location'] = 'Tables/ODS/employee_json_data_errors'
        fixed_row['file_type'] = row['file_type']
        fixed_row['was_error'] = 'EXTRA_FIXED'
        fixed_rows.append(fixed_row)
    
    return fixed_rows

fixed_extra = fix_extra_columns(df_extra)

if fixed_extra:
    schema = StructType([
        StructField("emp_id", StringType(), True),
        StructField("full_name", StringType(), True),
        StructField("fname", StringType(), True),
        StructField("lname", StringType(), True),
        StructField("birth_date", StringType(), True),
        StructField("dept_name", StringType(), True),
        StructField("country", StringType(), True),
        StructField("monthly_salary", StringType(), True),
        StructField("sex", StringType(), True),
        StructField("job_title", StringType(), True),
        StructField("employee_status", StringType(), True),
        StructField("education", StringType(), True),
        StructField("load_timestamp", StringType(), True),
        StructField("source_location", StringType(), True),
        StructField("file_type", StringType(), True),
        StructField("was_error", StringType(), True)
    ])
    
    df_fixed_extra = spark.createDataFrame(fixed_extra, schema)
    print(f"Fixed {df_fixed_extra.count()} extra column errors")
    
    print("\nFixed records:")
    display(df_fixed_extra.select("emp_id", "full_name", "job_title", "employee_status", "education").toPandas())

StatementMeta(, 51b91c34-088e-474b-8c39-afc2d171217f, 10, Finished, Available, Finished, False)

Fixed 5 extra column errors

Fixed records:


SynapseWidget(Synapse.DataFrame, 702adff8-fc4a-4f79-88f2-4ce8052e975b)

# 6: General Exploration for the data frame that we should combine

In [ ]:
df_fixed_manual.head()

StatementMeta(, 51b91c34-088e-474b-8c39-afc2d171217f, 27, Finished, Available, Finished, False)

Row(emp_id='100250', full_name='Heba Mostafa', fname='Heba', lname='Mostafa', birth_date=None, dept_name='SALES', country='Jordan', monthly_salary='44676', sex='Male', job_title='Supply Chain Coordinator', employee_status='on leave', education=None, load_timestamp='2026-07-18 10:39:02', source_location='Tables/ODS/employee_text_data_errors', file_type='txt', was_error='MISSING_FIXED')

In [ ]:
df_fixed_extra.head()

StatementMeta(, 51b91c34-088e-474b-8c39-afc2d171217f, 30, Finished, Available, Finished, False)

Row(emp_id='100777', full_name='Abdullah Davis', fname='Abdullah', lname='Davis', birth_date='1996-12-05', dept_name='human resources', country='QAT', monthly_salary='14131', sex='Male', job_title='Project Manager', employee_status='External', education=' PhD ', load_timestamp='2026-07-17 23:41:00', source_location='Files/employee_dirty_files_pre_ODS/employee_dirty_txt.txt', file_type='txt', was_error='EXTRA_FIXED')

In [ ]:
print("="*50)
print("DATA RECONCILIATION")
print("="*50)

print("\n📊 Error Type Breakdown:")
df_txt_errors.groupBy("error_type").count().show()

print("\n🔍 Let's examine the 7 error records:")
display(df_txt_errors.select("line_number", "error_type", "actual_fields", "original_line").toPandas())

print("\n" + "="*50)
print("FIXED RECORDS BREAKDOWN")
print("="*50)

print(f"df_fixed_extra count: {df_fixed_extra.count()} (EXTRA_COLUMNS fixed)")
print(f"df_fixed_manual count: {df_fixed_manual.count()} (MISSING_COLUMNS fixed)")

print("\n📊 Combined fixed records:")
print(f"Total fixed: {df_fixed_extra.count() + df_fixed_manual.count()}")
print(f"Should equal total errors: {df_txt_errors.count()}")
print(f"Difference: {df_txt_errors.count() - (df_fixed_extra.count() + df_fixed_manual.count())}")

print("\n" + "="*50)
print("FINAL RECONCILIATION")
print("="*50)

print(f"Original good: {df_txt_good.count()}")
print(f"Fixed extra: {df_fixed_extra.count()}")
print(f"Fixed manual: {df_fixed_manual.count()}")
print(f"Total combined: {df_txt_good.count() + df_fixed_extra.count() + df_fixed_manual.count()}")
print(f"Expected total: {df_txt_good.count() + df_txt_errors.count()}")

if df_txt_good.count() + df_fixed_extra.count() + df_fixed_manual.count() == df_txt_good.count() + df_txt_errors.count():
    print("✅ NUMBERS MATCH!")
else:
    print("❌ NUMBERS DON'T MATCH!")

StatementMeta(, 51b91c34-088e-474b-8c39-afc2d171217f, 13, Finished, Available, Finished, False)

DATA RECONCILIATION

📊 Error Type Breakdown:
+---------------+-----+
|     error_type|count|
+---------------+-----+
|  EXTRA_COLUMNS|    5|
|MISSING_COLUMNS|    2|
+---------------+-----+


🔍 Let's examine the 7 error records:


SynapseWidget(Synapse.DataFrame, 9b8edbc0-60ce-4315-bd45-5cb6b89f678d)


FIXED RECORDS BREAKDOWN


df_fixed_extra count: 5 (EXTRA_COLUMNS fixed)
df_fixed_manual count: 2 (MISSING_COLUMNS fixed)

📊 Combined fixed records:


Total fixed: 7
Should equal total errors: 7


Difference: 0

FINAL RECONCILIATION
Original good: 993
Fixed extra: 5
Fixed manual: 2


Total combined: 1000
Expected total: 1000


✅ NUMBERS MATCH!


# 7: Merging the fixed rows with the already good rows

In [ ]:
from pyspark.sql.functions import lit

# Add was_error column to df_txt_good
df_txt_good_with_flag = df_txt_good.withColumn("was_error", lit("NO SCHEMA ERROR"))

# Union all three dataframes
df_combined = df_txt_good_with_flag.unionByName(df_fixed_manual, allowMissingColumns=True) \
                                   .unionByName(df_fixed_extra, allowMissingColumns=True)


print(f"✅ Combined all records: {df_combined.count()}")
print(f"   Good records: {df_txt_good.count()}")
print(f"   Fixed manual: {df_fixed_manual.count()}")
print(f"   Fixed extra: {df_fixed_extra.count()}")

print("\n📊 Data quality breakdown:")
df_combined.groupBy("was_error").count().show()


StatementMeta(, 51b91c34-088e-474b-8c39-afc2d171217f, 32, Finished, Available, Finished, False)

✅ Combined all records: 1000


   Good records: 993
   Fixed manual: 2
   Fixed extra: 5

📊 Data quality breakdown:
+---------------+-----+
|      was_error|count|
+---------------+-----+
|NO SCHEMA ERROR|  993|
|  MISSING_FIXED|    2|
|    EXTRA_FIXED|    5|
+---------------+-----+


✅ Saved to ODS.CombinedEmployeeTextData


# 8: Saving the combined data to the ODS layer

In [ ]:
# Save df_combined to ODS schema in lakehouse
df_combined.write.format("delta").mode("overwrite").saveAsTable("ODS.employee_text_data_combined")

print(f"✅ Saved to ODS.employee_text_data_combined")
print(f"Total records: {df_combined.count()}")
print(f"Columns: {df_combined.columns}")

print("\n📊 Data quality breakdown:")
df_combined.groupBy("was_error").count().show()

print("\n📊 Sample data:")
display(df_combined.limit(5).toPandas())

StatementMeta(, 51b91c34-088e-474b-8c39-afc2d171217f, 33, Finished, Available, Finished, False)

✅ Saved to ODS.employee_text_data_combined


Total records: 1000
Columns: ['emp_id', 'full_name', 'fname', 'lname', 'birth_date', 'dept_name', 'country', 'monthly_salary', 'sex', 'job_title', 'employee_status', 'education', 'load_timestamp', 'source_location', 'file_type', 'was_error']

📊 Data quality breakdown:
+---------------+-----+
|      was_error|count|
+---------------+-----+
|NO SCHEMA ERROR|  993|
|  MISSING_FIXED|    2|
|    EXTRA_FIXED|    5|
+---------------+-----+


📊 Sample data:


SynapseWidget(Synapse.DataFrame, 05262b72-6d54-4b80-98ff-942454d4b2cc)